# TF-IDF and frozen-embedding detectors

Table 4 reports the mean and standard deviation over randomly selected
train/test splits. Each run fits the TF-IDF vectoriser on that run's training
prompts only. The corpus is the deduplicated union of the binary files, so a
prompt cannot sit on both sides of a split.

`NUM_RUNS` defaults to 30. Set it to 1 for a CPU smoke check.
The frozen-BERT section is the second half of this notebook; set
`RUN_BERT_EMBEDDINGS=0` to skip the download.

In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "code").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT / "code"))

from binary_data import load_clean_binary_corpus
from classical_experiments import run_matrix_model_runs, run_tfidf_runs
from latex_tables import export_results

NUM_RUNS = int(os.environ.get("NUM_RUNS", "30"))
corpus, report = load_clean_binary_corpus()
print(
    f"Corpus rows={len(corpus)} "
    f"shared_prompts_removed={report['shared_prompts_before']} "
    f"ambiguous_removed={report['removed_ambiguous_prompts']}"
)
tfidf_runs = run_tfidf_runs(corpus, num_runs=NUM_RUNS)
tfidf_paths = export_results("02_tfidf", "binary", tfidf_runs)
print(f"Wrote {tfidf_paths['tex']}")

## Frozen BERT embeddings

Embeddings are computed once. The classifier splits below are the supervised
part, and they are checked for prompt overlap before each fit.

In [ ]:
RUN_BERT_EMBEDDINGS = os.environ.get("RUN_BERT_EMBEDDINGS", "1") == "1"
if RUN_BERT_EMBEDDINGS:
    import numpy as np
    import torch
    from tqdm import tqdm
    from transformers import BertModel, BertTokenizer

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    bert_model = BertModel.from_pretrained("bert-base-uncased")
    bert_model.eval()

    def extract_bert_features(texts, tokenizer, model, max_length=128):
        embeddings = []
        for text in tqdm(texts, desc="Extracting BERT features"):
            tokens = tokenizer.encode_plus(
                str(text),
                max_length=max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            with torch.no_grad():
                output = model(
                    input_ids=tokens["input_ids"],
                    attention_mask=tokens["attention_mask"],
                )
            embeddings.append(output.last_hidden_state[:, 0, :].squeeze().numpy())
        return np.vstack(embeddings)

    features = extract_bert_features(corpus["prompt"], tokenizer, bert_model)
    embedding_runs = run_matrix_model_runs(
        features,
        corpus["prompt"],
        corpus["jailbreak"],
        features_name="BERT",
        num_runs=NUM_RUNS,
    )
    embedding_paths = export_results("02_bert_features", "binary", embedding_runs)
    print(f"Wrote {embedding_paths['tex']}")
else:
    print("Skipped frozen BERT embeddings.")